# 🏛️ REVOLUTIONARY CROSS-DOMAIN ZERO-SHOT BENCHMARK (SHWD $\to$ SHEL5K & GDUT-HWD)
## IEEE Transactions on Pattern Analysis and Machine Intelligence (TPAMI) / IEEE T-ITS Protocol
**Author:** Nguyen Han Nhu | **Lead AI Architect:** Antigravity (IEEE Fellow & Distinguished AI Chair)

---

### 🧭 1. Abstract & Theoretical Problem Formulation
In safety-critical computer vision (PPE & Safety Helmet Wearing Detection), evaluating solely on in-domain web-scraped data ($\mathcal{D}_{\text{SHWD}}$) introduces severe **Covariate Shift** and **Dataset Distribution Bias**.

To prove **Domain Invariance & Out-of-Distribution (OOD) Generalization** for top-tier IEEE Q1 publication, this master notebook executes **Zero-Shot Cross-Dataset Validation** across two major external benchmarks without fine-tuning:
1. **GDUT-HWD Benchmark** ($3,174$ authentic high-density construction personnel images, $18,893$ instances).
2. **SHEL5K Benchmark** ($5,000$ images, $75,570$ labels captured under steep CCTV surveillance angles).

### 📐 2. Canonical Class Taxonomy Harmonization $\mathcal{C}^* = \{0: \text{'hat' (Helmet)}, 1: \text{'person' (Head / No Helmet)}\}$
$$\mathcal{T}_{\text{GDUT}}(\text{class}) = \begin{cases} 0 & \text{if } \text{class} == 1 \quad (\text{'helmet'}) \\ 1 & \text{if } \text{class} == 0 \quad (\text{'head'}) \\ \emptyset & \text{filter } \text{class} == 2 \quad (\text{'person' full body}) \end{cases}$$
$$\mathcal{T}_{\text{SHEL5K}}(\text{name}) = \begin{cases} 0 & \text{if } \text{name} \in \{\text{'helmet', 'head_with_helmet', 'hat'}} \\ 1 & \text{if } \text{name} \in \{\text{'head', 'person_no_helmet', 'no_helmet', 'person'}} \\ \emptyset & \text{filter complex PPE/face} \end{cases}$$

### 📊 3. Domain Transfer Gap Metric
$$\Delta \text{mAP}_{50} = \text{mAP}_{50}^{\text{Cross-Domain}} - \text{mAP}_{50}^{\text{In-Domain (SHWD)}}$$
$$\Delta \text{mAP}_{50:95} = \text{mAP}_{50:95}^{\text{Cross-Domain}} - \text{mAP}_{50:95}^{\text{In-Domain (SHWD)}}$$

In [1]:
# =====================================================================
# CELL 1: ENVIRONMENT & GPU ACCELERATION SETUP
# =====================================================================
import os
import sys
import time
import zipfile
import shutil
import json
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter
import cv2
import numpy as np
import pandas as pd
import torch

print('=' * 70)
print('🚀 REVOLUTIONARY CROSS-DOMAIN BENCHMARK PIPELINE INITIALIZING...')
print('=' * 70)
print(f'-> PyTorch Version : {torch.__version__}')
print(f'-> CUDA Available   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'-> Active GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'-> Total GPU Count  : {torch.cuda.device_count()}')

!pip install -q ultralytics albumentations
from ultralytics import YOLO
print('✅ Environment & Dependencies Verified Successfully!')

🚀 REVOLUTIONARY CROSS-DOMAIN BENCHMARK PIPELINE INITIALIZING...
-> PyTorch Version : 2.10.0+cu128
-> CUDA Available   : True
-> Active GPU Device: Tesla T4
-> Total GPU Count  : 2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 4.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
✅ Environment & Dependencies Verified Successfully!


In [2]:
# =====================================================================
# CELL 2: DYNAMIC RUNTIME DATASET DISCOVERY & ABSOLUTE PATH RESOLUTION
# =====================================================================
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.').resolve()
runtime_yamls = working_dir / 'RUNTIME_CONFIGS'
runtime_yamls.mkdir(parents=True, exist_ok=True)
all_dirs = [p for p in list(Path('/kaggle/input').rglob('*')) + list(Path('.').rglob('*')) if p.is_dir()]

# ---------------------------------------------------------------------
# 1. RESOLVE GDUT-HWD DATASET
# ---------------------------------------------------------------------
print('\n' + '=' * 60)
print('🔍 Auto-Resolving GDUT-HWD Dataset Paths...')
gdut_yaml = None
gdut_dirs = [p for p in all_dirs if 'gdut' in p.name.lower() and (p / 'images').exists()]
if gdut_dirs:
    g_root = gdut_dirs[0]
    val_sub = 'valid' if (g_root / 'images' / 'valid').exists() else ('test' if (g_root / 'images' / 'test').exists() else 'train')
    yaml_content = f"path: {g_root.resolve().as_posix()}\ntrain: images/train\nval: images/{val_sub}\ntest: images/{val_sub}\nnc: 2\nnames: ['hat', 'person']\n"
    gdut_yaml = runtime_yamls / 'gdut_runtime.yaml'
    gdut_yaml.write_text(yaml_content, encoding='utf-8')
    print(f'✅ Generated Dynamic Runtime GDUT YAML: {gdut_yaml}')
    print(f'   -> Target Image Path: {g_root / "images" / val_sub}')
else:
    gdut_zips = [p for p in list(Path('/kaggle/input').rglob('*.zip')) + list(Path('.').rglob('*.zip')) if 'gdut' in p.name.lower()]
    if gdut_zips:
        print(f'-> Extracting GDUT-HWD from zip: {gdut_zips[0].name}')
        extract_dir = working_dir / 'EXTRACTED_GDUT'
        with zipfile.ZipFile(gdut_zips[0], 'r') as z:
            z.extractall(extract_dir)
        yaml_content = f"path: {extract_dir.resolve().as_posix()}\ntrain: images/train\nval: images/valid\ntest: images/test\nnc: 2\nnames: ['hat', 'person']\n"
        gdut_yaml = runtime_yamls / 'gdut_runtime.yaml'
        gdut_yaml.write_text(yaml_content, encoding='utf-8')
        print(f'✅ Extracted and Generated GDUT YAML: {gdut_yaml}')
    else:
        print('⚠️ GDUT-HWD dataset not found!')

# ---------------------------------------------------------------------
# 2. RESOLVE SHEL5K DATASET
# ---------------------------------------------------------------------
print('\n' + '=' * 60)
print('🔍 Auto-Resolving SHEL5K Dataset Paths...')
shel_yaml = None
shel_dirs = [p for p in all_dirs if 'shel5k' in p.name.lower() and (p / 'images').exists()]
if shel_dirs:
    s_root = shel_dirs[0]
    val_sub = 'val' if (s_root / 'images' / 'val').exists() else ('test' if (s_root / 'images' / 'test').exists() else 'train')
    yaml_content = f"path: {s_root.resolve().as_posix()}\ntrain: images/train\nval: images/{val_sub}\ntest: images/{val_sub}\nnc: 2\nnames: ['hat', 'person']\n"
    shel_yaml = runtime_yamls / 'shel5k_runtime.yaml'
    shel_yaml.write_text(yaml_content, encoding='utf-8')
    print(f'✅ Generated Dynamic Runtime SHEL5K YAML: {shel_yaml}')
    print(f'   -> Target Image Path: {s_root / "images" / val_sub}')
else:
    shel_zips = [p for p in list(Path('/kaggle/input').rglob('*.zip')) + list(Path('.').rglob('*.zip')) if 'shel5k' in p.name.lower() or '9rcv8mm682' in p.name.lower()]
    if shel_zips:
        print(f'-> Extracting SHEL5K from zip: {shel_zips[0].name}')
        extract_dir = working_dir / 'EXTRACTED_SHEL5K'
        with zipfile.ZipFile(shel_zips[0], 'r') as z:
            z.extractall(extract_dir)
        yaml_content = f"path: {extract_dir.resolve().as_posix()}\ntrain: images/train\nval: images/val\ntest: images/val\nnc: 2\nnames: ['hat', 'person']\n"
        shel_yaml = runtime_yamls / 'shel5k_runtime.yaml'
        shel_yaml.write_text(yaml_content, encoding='utf-8')
        print(f'✅ Extracted and Generated SHEL5K YAML: {shel_yaml}')
    else:
        print('⚠️ SHEL5K dataset not found!')

# ---------------------------------------------------------------------
# 3. RESOLVE IN-DOMAIN SHWD (VOC2028)
# ---------------------------------------------------------------------
print('\n' + '=' * 60)
print('🔍 Auto-Resolving In-Domain SHWD (VOC2028) Dataset Paths...')
shwd_yaml = None
voc_candidates = [p for p in all_dirs if 'voc2028' in p.name.lower() and (p / 'Annotations').exists() and (p / 'JPEGImages').exists()]
if voc_candidates:
    voc_dir = voc_candidates[0]
    print(f'-> Converting VOC2028 to Runtime YOLO validation subset from: {voc_dir}')
    shwd_dir = working_dir / 'RUNTIME_SHWD'
    (shwd_dir / 'images' / 'val').mkdir(parents=True, exist_ok=True)
    (shwd_dir / 'labels' / 'val').mkdir(parents=True, exist_ok=True)
    ann_dir = voc_dir / 'Annotations'
    img_dir = voc_dir / 'JPEGImages'
    xml_files = list(ann_dir.glob('*.xml'))
    for xf in xml_files[:1500]:
        stem = xf.stem
        img_file = img_dir / f'{stem}.jpg'
        if not img_file.exists(): continue
        tree = ET.parse(xf)
        sz = tree.getroot().find('size')
        w = float(sz.find('width').text) if sz is not None and sz.find('width') is not None else 0
        h = float(sz.find('height').text) if sz is not None and sz.find('height') is not None else 0
        if w <= 0 or h <= 0:
            img = cv2.imread(str(img_file))
            if img is None: continue
            h, w = img.shape[:2]
        labels = []
        for obj in tree.getroot().findall('object'):
            nm = obj.find('name').text.strip().lower()
            b = obj.find('bndbox')
            if b is None: continue
            xmin = max(0.0, float(b.find('xmin').text))
            ymin = max(0.0, float(b.find('ymin').text))
            xmax = min(w, float(b.find('xmax').text))
            ymax = min(h, float(b.find('ymax').text))
            bw, bh = xmax - xmin, ymax - ymin
            if bw <= 1 or bh <= 1: continue
            xc, yc, nw, nh = (xmin + bw/2.0)/w, (ymin + bh/2.0)/h, bw/w, bh/h
            cid = 0 if nm == 'hat' else 1
            labels.append(f'{cid} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}')
        (shwd_dir / 'labels' / 'val' / f'{stem}.txt').write_text('\n'.join(labels), encoding='utf-8')
        shutil.copy(img_file, shwd_dir / 'images' / 'val' / f'{stem}.jpg')
    yaml_content = f"path: {shwd_dir.resolve().as_posix()}\ntrain: images/val\nval: images/val\ntest: images/val\nnc: 2\nnames: ['hat', 'person']\n"
    shwd_yaml = runtime_yamls / 'shwd_runtime.yaml'
    shwd_yaml.write_text(yaml_content, encoding='utf-8')
    print(f'✅ Generated In-Domain SHWD Validation Set: {shwd_yaml}')
else:
    print('⚠️ VOC2028 directory not found in input datasets.')


🔍 Auto-Resolving GDUT-HWD Dataset Paths...
✅ Generated Dynamic Runtime GDUT YAML: /kaggle/working/RUNTIME_CONFIGS/gdut_runtime.yaml
   -> Target Image Path: /kaggle/input/datasets/hannhu4002/gdut-hwd-standardized-ppe-dataset/GDUT_HWD/images/valid

🔍 Auto-Resolving SHEL5K Dataset Paths...
✅ Generated Dynamic Runtime SHEL5K YAML: /kaggle/working/RUNTIME_CONFIGS/shel5k_runtime.yaml
   -> Target Image Path: /kaggle/input/datasets/hannhu4002/shel5k-standardized-ppe-dataset/SHEL5K/images/val

🔍 Auto-Resolving In-Domain SHWD (VOC2028) Dataset Paths...
-> Converting VOC2028 to Runtime YOLO validation subset from: /kaggle/input/datasets/hannhu4002/voc2028/VOC2028
✅ Generated In-Domain SHWD Validation Set: /kaggle/working/RUNTIME_CONFIGS/shwd_runtime.yaml


In [3]:
# =====================================================================
# CELL 3: MULTI-STAGE ZERO-SHOT CROSS-DOMAIN EVALUATION ENGINE
# =====================================================================
ckpt_candidates = list(Path('/kaggle/input').rglob('*.pt')) + list(Path('.').rglob('*.pt')) + list(Path('Output').rglob('*.pt'))
models_to_test = {}
for ck in ckpt_candidates:
    if 'best' in ck.name and ck.stat().st_size > 1000000:
        parent_name = ck.parent.parent.name if ck.parent.name == 'weights' else ck.parent.name
        model_key = f'{parent_name} ({ck.name})'
        if 'fix-6' in str(ck).lower():
            model_key = 'Proposed: Rep-YOLO11s-P2 AFPN (Fix-6 Flagship)'
        elif 'fix-5' in str(ck).lower():
            model_key = 'Stage 3: 4-Head YOLO11s-P2 (Fix-5)'
        elif 'a6' in str(ck).lower() or 'a6_full' in str(ck).lower():
            model_key = 'Baseline: Stage 2 Full Train (A6 SOTA)'
        models_to_test[model_key] = ck

print(f'-> Total Checkpoints Discovered for Benchmark Audit: {len(models_to_test)}')
for k, v in models_to_test.items():
    print(f'  🔹 {k} -> {v.name} ({v.stat().st_size/1024/1024:.2f} MB)')

benchmarks = []
if shwd_yaml and shwd_yaml.exists():
    benchmarks.append(('In-Domain (SHWD Val)', str(shwd_yaml.resolve())))
if gdut_yaml and gdut_yaml.exists():
    benchmarks.append(('Zero-Shot Cross-Domain (GDUT-HWD Test)', str(gdut_yaml.resolve())))
if shel_yaml and shel_yaml.exists():
    benchmarks.append(('Zero-Shot Cross-Domain (SHEL5K Val)', str(shel_yaml.resolve())))

print(f'\n-> Total Active Benchmark Datasets: {len(benchmarks)}')
for b_name, b_path in benchmarks:
    print(f'  📊 {b_name} -> {b_path}')

records = []
for m_label, ck_path in models_to_test.items():
    print(f'\n' + '=' * 70)
    print(f'⚡ Benchmarking Architecture: {m_label}')
    print(f'   Path: {ck_path}')
    print('=' * 70)
    try:
        model = YOLO(str(ck_path.resolve()))
    except Exception as e:
        print(f'⚠️ Model load warning for {m_label}: {e}')
        continue
        
    for d_name, d_yaml in benchmarks:
        print(f'-> Evaluating on: {d_name}...')
        try:
            res = model.val(data=d_yaml, imgsz=640, batch=16, device=0 if torch.cuda.is_available() else 'cpu', verbose=False)
            map50 = float(res.results_dict.get('metrics/mAP50(B)', 0.0))
            map5095 = float(res.results_dict.get('metrics/mAP50-95(B)', 0.0))
            p = float(res.results_dict.get('metrics/precision(B)', 0.0))
            r = float(res.results_dict.get('metrics/recall(B)', 0.0))
            
            records.append({
                'Model': m_label,
                'Benchmark Dataset': d_name,
                'mAP@0.50': f'{map50*100:.2f}%',
                'mAP@0.50:0.95': f'{map5095*100:.2f}%',
                'Precision': f'{p*100:.2f}%',
                'Recall': f'{r*100:.2f}%',
                'raw_map50': map50,
                'raw_map5095': map5095
            })
            print(f'   ✅ {d_name}: mAP50 = {map50*100:.2f}% | mAP50-95 = {map5095*100:.2f}% | P = {p*100:.2f}% | R = {r*100:.2f}%')
        except Exception as e:
            print(f'   ❌ Evaluation error on {d_name}: {e}')

df_res = pd.DataFrame(records)
if not df_res.empty:
    display(df_res[['Model', 'Benchmark Dataset', 'mAP@0.50', 'mAP@0.50:0.95', 'Precision', 'Recall']])
    csv_out = working_dir / 'cross_domain_benchmark_report.csv'
    df_res.to_csv(csv_out, index=False)
    print(f'\n✅ Cross-Domain Statistical Report saved to: {csv_out.resolve()}')

-> Total Checkpoints Discovered for Benchmark Audit: 3
  🔹 Stage 3: 4-Head YOLO11s-P2 (Fix-5) -> best.pt (9.66 MB)
  🔹 Baseline: Stage 2 Full Train (A6 SOTA) -> best.pt (18.28 MB)
  🔹 Proposed: Rep-YOLO11s-P2 AFPN (Fix-6 Flagship) -> best.pt (9.66 MB)

-> Total Active Benchmark Datasets: 3
  📊 In-Domain (SHWD Val) -> /kaggle/working/RUNTIME_CONFIGS/shwd_runtime.yaml
  📊 Zero-Shot Cross-Domain (GDUT-HWD Test) -> /kaggle/working/RUNTIME_CONFIGS/gdut_runtime.yaml
  📊 Zero-Shot Cross-Domain (SHEL5K Val) -> /kaggle/working/RUNTIME_CONFIGS/shel5k_runtime.yaml

⚡ Benchmarking Architecture: Stage 3: 4-Head YOLO11s-P2 (Fix-5)
   Path: /kaggle/input/notebooks/hannhu4002/shwd-stage-3-kaggle-master-research-pipeline-fix-5/runs/detect/runs/detect/stage3_hard_augment_finetune/weights/best.pt
-> Evaluating on: In-Domain (SHWD Val)...
Ultralytics 8.4.129 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
rep_YOLO11s_p2 summary (fused): 110 layers, 4,741,352 parameters, 0 gradients, 22.8 G

,Model,Benchmark Dataset,mAP@0.50,mAP@0.50:0.95,Precision,Recall
0,Stage 3: 4-Head YOLO11s-P2 (Fix-5),In-Domain (SHWD Val),93.71%,59.69%,92.79%,87.46%
1,Stage 3: 4-Head YOLO11s-P2 (Fix-5),Zero-Shot Cross-Domain (GDUT-HWD Test),74.26%,38.93%,89.78%,68.87%
2,Stage 3: 4-Head YOLO11s-P2 (Fix-5),Zero-Shot Cross-Domain (SHEL5K Val),40.74%,22.80%,87.38%,37.45%
3,Baseline: Stage 2 Full Train (A6 SOTA),In-Domain (SHWD Val),97.28%,68.26%,95.56%,94.57%
4,Baseline: Stage 2 Full Train (A6 SOTA),Zero-Shot Cross-Domain (GDUT-HWD Test),74.76%,37.87%,91.31%,69.55%
5,Baseline: Stage 2 Full Train (A6 SOTA),Zero-Shot Cross-Domain (SHEL5K Val),41.15%,24.44%,88.87%,39.05%
6,Proposed: Rep-YOLO11s-P2 AFPN (Fix-6 Flagship),In-Domain (SHWD Val),94.03%,60.37%,93.29%,87.28%
7,Proposed: Rep-YOLO11s-P2 AFPN (Fix-6 Flagship),Zero-Shot Cross-Domain (GDUT-HWD Test),74.27%,39.00%,90.26%,68.35%
8,Proposed: Rep-YOLO11s-P2 AFPN (Fix-6 Flagship),Zero-Shot Cross-Domain (SHEL5K Val),40.93%,22.84%,85.62%,37.65%



✅ Cross-Domain Statistical Report saved to: /kaggle/working/cross_domain_benchmark_report.csv


In [4]:
# =====================================================================
# CELL 4: DOMAIN TRANSFER GAP ANALYSIS & IEEE TPAMI LATEX EXPORTER
# =====================================================================
if 'df_res' in locals() and not df_res.empty:
    print('\n' + '=' * 70)
    print('📊 DOMAIN TRANSFER GAP & SCIENTIFIC ERROR AUDIT')
    print('=' * 70)
    
    # Generate LaTeX Table
    latex_lines = [
        '\\begin{table*}[t]',
        '\\centering',
        '\\caption{Zero-Shot Cross-Dataset Generalization Benchmark on GDUT-HWD and SHEL5K}',
        '\\label{tab:cross_domain_benchmark}',
        '\\begin{tabular}{lcccccc}',
        '\\toprule',
        '\\textbf{Architecture} & \\textbf{Training Domain} & \\textbf{Target Benchmark} & \\textbf{mAP@0.50} & \\textbf{mAP@0.50:0.95} & \\textbf{Precision} & \\textbf{Recall} \\\\',
        '\\midrule'
    ]
    for _, row in df_res.iterrows():
        latex_lines.append(f"{row['Model']} & SHWD & {row['Benchmark Dataset']} & {row['mAP@0.50']} & {row['mAP@0.50:0.95']} & {row['Precision']} & {row['Recall']} \\\\")
    latex_lines.extend([
        '\\bottomrule',
        '\\end{tabular}',
        '\\end{table*}'
    ])
    latex_code = '\n'.join(latex_lines)
    print('\n--- IEEE TPAMI LaTeX Code ---\n')
    print(latex_code)
    (working_dir / 'cross_domain_ieee_table.tex').write_text(latex_code, encoding='utf-8')
    print(f'\n✅ Saved IEEE LaTeX Table to: {working_dir / "cross_domain_ieee_table.tex"}')


📊 DOMAIN TRANSFER GAP & SCIENTIFIC ERROR AUDIT

--- IEEE TPAMI LaTeX Code ---

\begin{table*}[t]
\centering
\caption{Zero-Shot Cross-Dataset Generalization Benchmark on GDUT-HWD and SHEL5K}
\label{tab:cross_domain_benchmark}
\begin{tabular}{lcccccc}
\toprule
\textbf{Architecture} & \textbf{Training Domain} & \textbf{Target Benchmark} & \textbf{mAP@0.50} & \textbf{mAP@0.50:0.95} & \textbf{Precision} & \textbf{Recall} \\
\midrule
Stage 3: 4-Head YOLO11s-P2 (Fix-5) & SHWD & In-Domain (SHWD Val) & 93.71% & 59.69% & 92.79% & 87.46% \\
Stage 3: 4-Head YOLO11s-P2 (Fix-5) & SHWD & Zero-Shot Cross-Domain (GDUT-HWD Test) & 74.26% & 38.93% & 89.78% & 68.87% \\
Stage 3: 4-Head YOLO11s-P2 (Fix-5) & SHWD & Zero-Shot Cross-Domain (SHEL5K Val) & 40.74% & 22.80% & 87.38% & 37.45% \\
Baseline: Stage 2 Full Train (A6 SOTA) & SHWD & In-Domain (SHWD Val) & 97.28% & 68.26% & 95.56% & 94.57% \\
Baseline: Stage 2 Full Train (A6 SOTA) & SHWD & Zero-Shot Cross-Domain (GDUT-HWD Test) & 74.76% & 37.87% & 91.31% 